In [1]:
import os
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from sklearn.metrics import classification_report
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import WeightedRandomSampler

In [2]:
# Move up directories until we find the 'data' folder
while not Path("data").exists() and len(Path.cwd().parts) > 1:
    os.chdir("..")
print(f"Working directory set to: {os.getcwd()}")

✅ Working directory set to: E:\CS\SEM 6\DL\Mini Project\Trash-Classification-System


In [3]:

# =========================
# 2. Load Config
# =========================
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

SPLITS_CSV   = cfg["data"]["splits_csv"]
IMG_SIZE     = 224
NUM_CLASSES  = cfg["data"]["num_classes"]
BATCH_SIZE   = 16   # smaller for CPU
NUM_EPOCHS   = 5    # as you requested
LR           = 1e-4

DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

Using device: cpu


In [4]:

# =========================
# 3. Dataset Class
# =========================
class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "filepath"]
        label = self.df.loc[idx, "main_class"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(label)   
        label = torch.tensor(label, dtype=torch.long)


        return image, label


In [5]:

# =========================
# 4. Transforms
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


In [6]:

# =========================
# 5. Load Data
# =========================
df = pd.read_csv(SPLITS_CSV)

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]

# Compute class weights
labels = train_df["class_idx"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class Weights:", class_weights)

train_dataset = CustomDataset(train_df, train_transform)
val_dataset   = CustomDataset(val_df, val_transform)

# Count samples per class
class_counts = train_df["class_idx"].value_counts().sort_index().values

# Weight for each class
class_weights_np = 1.0 / class_counts

# Assign weight to each sample
sample_weights = class_weights_np[train_df["class_idx"].values]
sample_weights = sample_weights.copy()

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Data loaded:", len(train_dataset), "train,", len(val_dataset), "val")


Class Weights: tensor([0.3846, 0.8333, 2.5000, 1.2500, 1.6667, 2.5000])
Data loaded: 10500 train, 2250 val


In [7]:
print(train_df.columns)

Index(['filepath', 'main_class', 'class_idx', 'split'], dtype='str')


In [8]:

# =========================
# 6. Model
# =========================
model = timm.create_model("mobilenetv3_large_100", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
model = model.to(DEVICE)


In [9]:

# =========================
# 7. Loss & Optimizer
# =========================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LR)


In [10]:

# =========================
# 8. Training Loop
# =========================
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ---- VALIDATION ----
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = correct / total
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved!")



Epoch 1/5


  0%|                                                                                          | 0/657 [00:00<?, ?it/s]


ValueError: invalid literal for int() with base 10: 'Organic'

In [ ]:

# =========================
# 9. Final Evaluation
# =========================
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))